<a href="https://colab.research.google.com/github/fborrasumh/optimizia/blob/main/cuadernos/02-descomposicion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# OptimizIA · Cuaderno 2: Técnicas de descomposición

**Técnicas Avanzadas de Optimización (8265)** · Máster en Estadística Computacional y Ciencia de Datos para la Toma de Decisiones · UMH.
Cuaderno de práctica de la app OptimizIA (Fernando Borrás Rocher, Mercedes Landete Ruiz y Juan Francisco Monge Ivars). Licencia MIT.

**Resultados de aprendizaje que trabaja:** RA3 (elegir el método de descomposición adecuado a la estructura) y RA4 (aplicarlo: generación de columnas, relajación lagrangiana, Benders).

Tres métodos, tres estructuras: **muchas variables** (generación de columnas), **restricciones que complican** (relajación lagrangiana) y **variables que complican** (Benders).

> Cómo usarlo: ejecuta las celdas en orden (Mayús+Intro). Los ejercicios están marcados con ✏️. Las celdas «Solución» están plegadas: despliégalas solo después de intentarlo.
> Las instancias son **pequeñas y ficticias**, pensadas para ver el mecanismo; los tiempos que veas dependen de la máquina de Colab.

In [ ]:
# Instala OR-Tools si no está (en Colab tarda unos segundos)
try:
    import ortools
except ImportError:
    %pip install -q ortools
    import ortools
print('OR-Tools', ortools.__version__)

## 1. Generación de columnas: el problema del corte (*cutting stock*)

Rollos de ancho $W$; pedidos de $d_k$ piezas de ancho $w_k$. Cada **patrón** $p$ (cuántas piezas de cada tipo salen de un rollo, $a_{kp}$) es una columna. Hay demasiados patrones para enumerarlos: el **maestro restringido** trabaja con unos pocos, y el **subproblema de precios** (una mochila con los duales $\pi_k$) busca un patrón con coste reducido $1-\sum_k \pi_k a_k < 0$.

In [ ]:
from ortools.linear_solver import pywraplp
W = 100; w = [45, 36, 31, 14]; d = [97, 610, 395, 211]

def mochila_entera(valores, pesos, cap):
    # Mochila entera sin límite (programación dinámica): máximo Σ valor·a con Σ peso·a ≤ cap
    best = [0.0] * (cap + 1); elec = [None] * (cap + 1)
    for c in range(1, cap + 1):
        best[c], elec[c] = best[c - 1], None
        for k, (vk, pk) in enumerate(zip(valores, pesos)):
            if pk <= c and best[c - pk] + vk > best[c] + 1e-12:
                best[c], elec[c] = best[c - pk] + vk, k
    a, c = [0] * len(pesos), cap
    while c > 0:
        if elec[c] is None: c -= 1
        else: a[elec[c]] += 1; c -= pesos[elec[c]]
    return best[cap], a

def maestro(patrones, entero=False):
    s = pywraplp.Solver.CreateSolver('SCIP' if entero else 'GLOP')
    lam = [s.IntVar(0, s.infinity(), '') if entero else s.NumVar(0, s.infinity(), '') for _ in patrones]
    filas = [s.Add(sum(p[k] * lam[q] for q, p in enumerate(patrones)) >= d[k]) for k in range(len(w))]
    s.Minimize(sum(lam)); s.Solve()
    return s.Objective().Value(), [f.dual_value() for f in filas] if not entero else None, [l.solution_value() for l in lam]

# Patrones iniciales: homogéneos (solo un tipo de pieza)
patrones = [[(W // w[k]) if j == k else 0 for j in range(len(w))] for k in range(len(w))]
for it in range(1, 50):
    z, pi, _ = maestro(patrones)
    valor, a = mochila_entera(pi, w, W)
    cr = 1 - valor
    print(f'it {it:2d}: z_maestro = {z:9.3f} · mejor coste reducido = {cr:+.4f} · patrón {a}')
    if cr > -1e-9: break
    patrones.append(a)
z_ent, _, lam = maestro(patrones, entero=True)
print(f'\nCota de la relajación (z_LP) = {z:.3f} → al menos {int(-(-z // 1))} rollos')
print(f'Maestro entero con las columnas generadas: {z_ent:.0f} rollos (solución factible; el óptimo garantizado exigiría branch-and-price)')

✏️ **Ejercicio 1.** Calcula **a mano** el coste reducido del patrón $(0,2,0,2)$ con los duales de la última iteración (`pi`). ¿Por qué el algoritmo ya no lo añade?

In [ ]:
#@title Solución del ejercicio 1 { display-mode: "form" }
patron = [0, 2, 0, 2]
assert sum(wk * ak for wk, ak in zip(w, patron)) <= W, 'el patrón no cabe'
cr = round(1 - sum(p * a for p, a in zip(pi, patron)), 9) + 0.0
print(f'coste reducido = {cr:+.4f} (≥ 0: ya no mejora el maestro; lo añadimos en la iteración 1 y sus duales se ajustaron)')

## 2. Relajación lagrangiana y método del subgradiente

Localización sin capacidades (UFL, cuaderno 1). Relajamos las restricciones de asignación $\sum_i x_{ij}=1$ con multiplicadores $\lambda_j$ (libres de signo):

$$L(\lambda)=\sum_j \lambda_j + \sum_i \min\Big(0,\; f_i + \sum_j \min(0,\, c_{ij}-\lambda_j)\Big)$$

El subproblema se resuelve **por inspección** (cada planta por separado). $L(\lambda)$ es una cota inferior para cualquier $\lambda$; el subgradiente la maximiza.

In [ ]:
import random
def instancia_ufl(m=8, n=25, semilla=1):
    rnd = random.Random(semilla)
    P = [(rnd.random(), rnd.random()) for _ in range(m)]; C = [(rnd.random(), rnd.random()) for _ in range(n)]
    f = [round(rnd.uniform(3, 6), 2) for _ in range(m)]
    c = [[round(10 * ((P[i][0]-C[j][0])**2 + (P[i][1]-C[j][1])**2) ** .5, 2) for j in range(n)] for i in range(m)]
    return f, c
f, c = instancia_ufl(); m, n = len(f), len(c[0])

def L(lam):
    val, y, x = sum(lam), [0]*m, [[0]*n for _ in range(m)]
    for i in range(m):
        r = f[i] + sum(min(0, c[i][j] - lam[j]) for j in range(n))
        if r < 0:
            val += r; y[i] = 1
            for j in range(n): x[i][j] = 1 if c[i][j] - lam[j] < 0 else 0
    return val, y, x

def heuristica(y):   # solución factible: abrir las plantas de y (o la mejor si no hay) y asignar cada cliente a la más barata
    A = [i for i in range(m) if y[i]] or [min(range(m), key=lambda i: f[i] + sum(c[i]))]
    return sum(f[i] for i in A) + sum(min(c[i][j] for i in A) for j in range(n))

lam = [min(c[i][j] for i in range(m)) for j in range(n)]
LB, UB, paso = -1e9, 1e9, 2.0
for k in range(300):
    val, y, x = L(lam); LB = max(LB, val); UB = min(UB, heuristica(y))
    g = [1 - sum(x[i][j] for i in range(m)) for j in range(n)]     # subgradiente
    if all(gj == 0 for gj in g): break
    t = paso * (UB - val) / sum(gj * gj for gj in g)
    lam = [lam[j] + t * g[j] for j in range(n)]
    if k % 30 == 29: paso /= 2
    if k % 50 == 0: print(f'it {k:3d}: L(λ) = {val:8.3f} · mejor cota = {LB:8.3f} · mejor factible = {UB:8.3f}')
print(f'\nTras {k+1} iteraciones: cota lagrangiana {LB:.3f} ≤ óptimo ≤ {UB:.3f} (gap {abs(100*(UB-LB)/UB):.2f} %)')

✏️ **Ejercicio 2.** Por el teorema de Geoffrion, como el subproblema tiene la **propiedad de integralidad**, la mejor cota lagrangiana coincide con la relajación lineal de la formulación fuerte. Compruébalo resolviendo esa relajación con GLOP.

In [ ]:
#@title Solución del ejercicio 2 { display-mode: "form" }
s = pywraplp.Solver.CreateSolver('GLOP')
y = [s.NumVar(0, 1, '') for _ in range(m)]; x = [[s.NumVar(0, 1, '') for _ in range(n)] for _ in range(m)]
for j in range(n): s.Add(sum(x[i][j] for i in range(m)) == 1)
for i in range(m):
    for j in range(n): s.Add(x[i][j] <= y[i])
s.Minimize(sum(f[i]*y[i] for i in range(m)) + sum(c[i][j]*x[i][j] for i in range(m) for j in range(n))); s.Solve()
print(f'z_LP fuerte = {s.Objective().Value():.3f} · mejor cota lagrangiana = {LB:.3f} (el subgradiente se acerca, no siempre la alcanza exactamente)')

## 3. Descomposición de Benders

Localización **con** capacidades: las $y_i$ (abrir) son las variables que complican. Fijadas, lo que queda es un problema de transporte (lineal). El maestro decide $y$ y una estimación $\eta$ del coste de transporte; cada subproblema devuelve un **corte de optimalidad** construido con sus duales.

Para no tratar cortes de factibilidad, exigimos en el maestro capacidad abierta suficiente: $\sum_i Q_i y_i \ge \sum_j d_j$ (así el subproblema siempre es factible).

In [ ]:
rnd = random.Random(5)
m, n = 5, 12
d = [rnd.randint(5, 20) for _ in range(n)]; Q = [rnd.randint(40, 80) for _ in range(m)]
F = [rnd.randint(80, 160) for _ in range(m)]; c = [[rnd.randint(1, 12) for _ in range(n)] for _ in range(m)]

def subproblema(ybar):
    s = pywraplp.Solver.CreateSolver('GLOP')
    x = [[s.NumVar(0, s.infinity(), '') for _ in range(n)] for _ in range(m)]
    dem = [s.Add(sum(x[i][j] for i in range(m)) >= d[j]) for j in range(n)]          # duales u_j ≥ 0
    cap = [s.Add(sum(x[i][j] for j in range(n)) <= Q[i] * ybar[i]) for i in range(m)]  # duales v_i ≤ 0
    s.Minimize(sum(c[i][j] * x[i][j] for i in range(m) for j in range(n)))
    assert s.Solve() == pywraplp.Solver.OPTIMAL
    return s.Objective().Value(), [r.dual_value() for r in dem], [r.dual_value() for r in cap]

cortes, LB, UB = [], -1e9, 1e9
for it in range(1, 30):
    M = pywraplp.Solver.CreateSolver('SCIP')
    y = [M.BoolVar(f'y{i}') for i in range(m)]; eta = M.NumVar(0, M.infinity(), 'eta')
    M.Add(sum(Q[i] * y[i] for i in range(m)) >= sum(d))
    for (u, v) in cortes:   # η ≥ Σ_j u_j d_j + Σ_i v_i Q_i y_i
        M.Add(eta >= sum(u[j] * d[j] for j in range(n)) + sum(v[i] * Q[i] * y[i] for i in range(m)))
    M.Minimize(sum(F[i] * y[i] for i in range(m)) + eta); M.Solve()
    ybar = [round(yi.solution_value()) for yi in y]; LB = M.Objective().Value()
    zsub, u, v = subproblema(ybar)
    UB = min(UB, sum(F[i] * ybar[i] for i in range(m)) + zsub)
    print(f'it {it:2d}: y = {ybar} · cota inferior = {LB:8.2f} · mejor factible = {UB:8.2f}')
    if UB - LB < 1e-6: break
    cortes.append((u, v))
print(f'\nBenders termina con óptimo {UB:.2f} tras {it} iteraciones')

✏️ **Ejercicio 3.** Comprueba el resultado resolviendo el modelo completo (monolítico) con SCIP. Después, discute: ¿en qué tipo de instancias compensaría Benders frente al modelo completo?

In [ ]:
#@title Solución del ejercicio 3 { display-mode: "form" }
s = pywraplp.Solver.CreateSolver('SCIP')
y = [s.BoolVar('') for _ in range(m)]; x = [[s.NumVar(0, s.infinity(), '') for _ in range(n)] for _ in range(m)]
for j in range(n): s.Add(sum(x[i][j] for i in range(m)) >= d[j])
for i in range(m): s.Add(sum(x[i][j] for j in range(n)) <= Q[i] * y[i])
s.Minimize(sum(F[i]*y[i] for i in range(m)) + sum(c[i][j]*x[i][j] for i in range(m) for j in range(n))); s.Solve()
print('Monolítico:', round(s.Objective().Value(), 2), '· Benders:', round(UB, 2))
# Compensa cuando, fijadas las y, el subproblema se separa en muchos subproblemas independientes (p. ej. escenarios
# de programación estocástica, Birge y Louveaux) o tiene estructura que un algoritmo especializado resuelve muy rápido.

## 🎯 Tu instancia personal y tu entrega

1. En OptimizIA, ve a **Cuadernos → Tu instancia personal** y pulsa **Copiar**.
2. Sustituye la línea `INSTANCIA = None` de la celda siguiente por la que has copiado y ejecuta las dos celdas.
3. Copia la línea `ENTREGA = {…}` que aparece y pégala en OptimizIA: la app recalcula tu instancia con su propio solver y valida tu resultado.

> ✏️ Antes de ejecutar, **predice**: ¿cuántas iteraciones de generación de columnas necesitará tu instancia? ¿Cuántas de Benders?

In [ ]:
INSTANCIA = None   # ← sustituye esta línea por la que copias de OptimizIA

In [ ]:
import json
def cg_corte(W, w, d):
    pats = [[(W // w[k]) if j == k else 0 for j in range(len(w))] for k in range(len(w))]
    while True:
        s = pywraplp.Solver.CreateSolver('GLOP'); lam = [s.NumVar(0, s.infinity(), '') for _ in pats]
        filas = [s.Add(sum(p[k] * lam[q] for q, p in enumerate(pats)) >= d[k]) for k in range(len(w))]
        s.Minimize(sum(lam)); s.Solve(); pi = [f.dual_value() for f in filas]
        valor, a = mochila_entera(pi, w, W)
        if 1 - valor > -1e-9: return s.Objective().Value(), len(pats) - len(w) + 1
        pats.append(a)
def cfl_completo(I):
    m, n, d, Q, F, c = I['m'], I['n'], I['d'], I['Q'], I['F'], I['c']
    s = pywraplp.Solver.CreateSolver('SCIP'); y = [s.BoolVar('') for _ in range(m)]; x = [[s.NumVar(0, s.infinity(), '') for _ in range(n)] for _ in range(m)]
    for j in range(n): s.Add(sum(x[i][j] for i in range(m)) >= d[j])
    for i in range(m): s.Add(sum(x[i][j] for j in range(n)) <= Q[i] * y[i])
    s.Minimize(sum(F[i]*y[i] for i in range(m)) + sum(c[i][j]*x[i][j] for i in range(m) for j in range(n))); s.Solve()
    return s.Objective().Value()
if INSTANCIA:
    cs, cf = INSTANCIA['corte'], INSTANCIA['cfl']
    z_cs, its = cg_corte(cs['W'], cs['w'], cs['d'])
    print(f'Generación de columnas: z_LP = {z_cs:.3f} en {its} iteraciones')
    res = {'z_lp_corte': z_cs, 'z_cfl': cfl_completo(cf)}
    print('ENTREGA =', json.dumps({'app': 'optimizia', 'cuaderno': 2, 'semilla': INSTANCIA['semilla'], 'res': res}))
    print('✏️ Reto: resuelve también tu CFL con tu propio Benders (sección 3) y comprueba que obtienes el mismo z_cfl.')
else:
    print('Pega primero tu INSTANCIA en la celda anterior.')

## Para llevarte
| Estructura del problema | Método natural |
|---|---|
| Muchísimas variables, pocas restricciones (patrones, rutas) | Generación de columnas / Dantzig-Wolfe |
| Unas pocas restricciones «de enlace» complican un problema fácil | Relajación lagrangiana |
| Unas pocas variables «complicantes»; fijadas, el resto es fácil o separable | Benders |